# FINAL V3 — live partial test monitor

Run this in a separate Colab while the main FINAL V3 notebook is training. It **does not train or modify models**. It only scans completed checkpoints and already-saved per-seed test-evaluation JSON files.

> Partial means/SDs are diagnostic until all five training seeds are present. Do not use partial test performance to retune the protocol.


In [ ]:
from pathlib import Path
import json, re, os, sys
import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = 'google.colab' in sys.modules or os.environ.get('COLAB_GPU') is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path(os.environ.get('DISSERTATION_DRIVE_ROOT', Path.cwd()))

RESULTS_ROOT = DRIVE_ROOT / 'dissertation' / 'final_four_model_all_manifolds_v1'
RUN_DIRNAME = 'final_v3_50k_cosine'
EXPECTED_SEEDS = {101, 102, 103, 104, 105}
print('Results root:', RESULTS_ROOT)
print('Monitoring:', RUN_DIRNAME)


In [ ]:
def scan_current_results():
    checkpoint_rows = []
    metric_rows = []

    for p in RESULTS_ROOT.glob(f'*/*/*/{RUN_DIRNAME}/seed_*.pt'):
        m = re.search(r'seed_(\\d+)\\.pt$', p.name)
        if not m:
            continue
        checkpoint_rows.append({
            'manifold': p.parents[3].name,
            'dataset': p.parents[2].name,
            'method': p.parents[1].name,
            'seed': int(m.group(1)),
            'checkpoint': str(p),
        })

    for p in RESULTS_ROOT.glob(f'*/*/*/{RUN_DIRNAME}/seed_*_test_evaluation.json'):
        m = re.search(r'seed_(\\d+)_test_evaluation\\.json$', p.name)
        if not m:
            continue
        with open(p, 'r') as f:
            ev = json.load(f)
        for metric, vals in ev.get('summary', {}).items():
            metric_rows.append({
                'manifold': p.parents[3].name,
                'dataset': p.parents[2].name,
                'method': p.parents[1].name,
                'seed': int(m.group(1)),
                'metric': metric,
                'value': float(vals['mean']),
                'std_test_noise': float(vals.get('std_noise', np.nan)),
                'best_step': ev.get('_best_validation_step', np.nan),
                'best_val': ev.get('_best_validation_score', np.nan),
            })

    checkpoints = pd.DataFrame(checkpoint_rows)
    metrics = pd.DataFrame(metric_rows)
    return checkpoints, metrics

checkpoints, metrics = scan_current_results()

print(f'Completed checkpoints: {len(checkpoints)}')
if len(metrics):
    n_tested = metrics[['manifold','dataset','method','seed']].drop_duplicates().shape[0]
else:
    n_tested = 0
print(f'Completed test evaluations: {n_tested}')

if len(checkpoints):
    progress = (checkpoints.groupby(['manifold','dataset','method'])['seed']
                .nunique().rename('n_completed').reset_index())
    progress['progress'] = progress['n_completed'].astype(str) + '/5'
    display(progress.sort_values(['manifold','dataset','method']))
else:
    print('No FINAL V3 checkpoints found yet.')


In [ ]:
# Partial test-performance summary. Rerun this cell whenever you want to refresh.
checkpoints, metrics = scan_current_results()

if metrics.empty:
    print('No test evaluations saved yet.')
else:
    partial = (metrics.groupby(['manifold','dataset','method','metric'], as_index=False)
               .agg(mean=('value','mean'),
                    sd=('value', lambda x: x.std(ddof=1) if len(x) > 1 else np.nan),
                    n_seeds=('seed','nunique')))

    key_metrics = [
        'ambient_mmd_sq_raw',
        'ambient_mmd_sq_projected',
        'intrinsic_discrepancy_sq_projected',
        'support_mean',
    ]
    key = partial[partial['metric'].isin(key_metrics)].copy()
    key['mean ± sd'] = key.apply(
        lambda r: f"{r['mean']:.6g} ± {r['sd']:.3g}" if np.isfinite(r['sd']) else f"{r['mean']:.6g} (n=1)",
        axis=1
    )
    wide = key.pivot_table(
        index=['manifold','dataset','method','n_seeds'],
        columns='metric', values='mean ± sd', aggfunc='first'
    ).reset_index()
    display(wide.sort_values(['manifold','dataset','method']))

    # Best validation steps for completed/tested seeds.
    best = (metrics[['manifold','dataset','method','seed','best_step','best_val']]
            .drop_duplicates()
            .sort_values(['manifold','dataset','method','seed']))
    display(best)
